In [1]:
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun

### Used the Wikipedia inbuilt tool

In [2]:
#Now let's build the wrapper for the tools above
wikipedia_api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)

#run the api wrapper 
wikipedia_tool = WikipediaQueryRun(api_wrapper = wikipedia_api_wrapper)

wikipedia_tool.name


'wikipedia'

In [3]:
wikipedia_tool.run("what is langsmith?")

'No good Wikipedia Search Result was found'

### use the Arxiv inbuilt tool

In [4]:
#buit the api wrapper
arxiv_api_wrapper = ArxivAPIWrapper(top_k_results= 1, doc_content_chars_max=250)

#run the api wrapper

arxiv_tool = ArxivQueryRun(api_wrapper = arxiv_api_wrapper)

arxiv_tool.name

'arxiv'

In [5]:
#test the arxiv tool
arxiv_tool.run("what is langsmith?")

'Published: 2018-05-17\nTitle: What is "fundamental"?\nAuthors: Matt Visser\nSummary: Our collective views regarding the question "what is fundamental?" are continually evolving. These ontological shifts in what we regard as fundamental are largely drive'

In [6]:
#combining the two tools
tools = [wikipedia_tool, arxiv_tool]

### Create your own custom tools

In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

hf_api_key = os.getenv("HF_API_KEY")

### custom tools [RAG TOOLS]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

/home/aljebra/Generative AI tutorial/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-18 16:49:16.710645: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
USER_AGENT environment variable not set, consider setting it to identify your requests.


### Loading a web page and put it in vector store 

In [8]:
#load the webpage 
loader = WebBaseLoader(web_paths = ["https://docs.smith.langchain.com/"])

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

split_documents = text_splitter.split_documents(documents)

vectorstore = FAISS.from_documents(split_documents, HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-V2'))

retriever = vectorstore.as_retriever()

retriever

/home/aljebra/Generative AI tutorial/venv/lib/python3.10/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x719720617280>, search_kwargs={})

In [9]:
retriever.invoke("what is langsmith")

[Document(id='4db84b11-e844-43a5-a62b-65442910cd84', metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KSupportGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewPlansCreate an account and API keyAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingSet up resource tagsUser managementAdditional resourcesPolly (Beta)Data managementAccess control & AuthenticationScalability & resilienceFAQsRegions FAQPricing FAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith provides tools for developing, debugging, and deploying LLM applications.\nIt helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place.\nLan

### convert the retriever into a tool (in other to use it as a tool)

In [10]:
from langchain_core.tools import create_retriever_tool

In [11]:
retriever_tool = create_retriever_tool(retriever, 'langsmith_retriever', 'Retrieves information from langsmith')

retriever_tool.name

'langsmith_retriever'

In [12]:
retriever_tool.invoke("what is langsmith")

'LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KSupportGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewPlansCreate an account and API keyAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingSet up resource tagsUser managementAdditional resourcesPolly (Beta)Data managementAccess control & AuthenticationScalability & resilienceFAQsRegions FAQPricing FAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith provides tools for developing, debugging, and deploying LLM applications.\nIt helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place.\nLangSmith is framework agnostic, so you can use it with or without LangChain’s open-source libraries\nlangchain and langgraph.\n\nLangSmith is framework agnostic, so you can use it with or wit

In [13]:
#update the list of tools
tools = [ wikipedia_tool, arxiv_tool, retriever_tool]

tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/home/aljebra/Generative AI tutorial/venv/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith_retriever', description='Retrieves information from langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x7197206069e0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x719720606a70>)]

### Run tools with Agent and LLM models

In [14]:
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")


model = ChatGroq(model='llama-3.1-8b-instant', api_key=groq_api_key)


model


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x71971af12da0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x71971af12cb0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [15]:
model.invoke("what is langsmith?")

AIMessage(content='LangSmith is a language model development platform. It focuses on allowing users to create, train, and deploy their own custom conversational AI models. The platform supports a range of tasks from chatbots and voice assistants to language translation and text summarization. LangSmith operates on pre-trained models and supports customization with user-defined datasets for fine-tuning and personalization of AI models.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 40, 'total_tokens': 116, 'completion_time': 0.21026499, 'completion_tokens_details': None, 'prompt_time': 0.001904081, 'prompt_tokens_details': None, 'queue_time': 0.088457607, 'total_time': 0.212169071}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019bd1cc-2f79-7500-b638-9b01fd110043-0', usage_metadata={'input_tok

### prompt template

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [17]:
prompt = ChatPromptTemplate.from_messages(
    [
        ('system', """You are a helpful ai assistance that can use tool to answer question.
         You have access to:
         - langsmith_retriever : for information about langsmith from its official docs
         - wikipedia: for general encyclopaedia information
         - arxiv : for acedemic paper and research

         when answering questions:
         1. use the most relevant tool for the queries.
         2. if one tool doesn't provide good result try another 
         3. provide a clear and comprehensive answer based on the information received
         4. if you can't find relevant answer, say so clearly."""
         
         ),
        ('human', '{input}'),
        MessagesPlaceholder("agent_scratchpad")
    ]
)

### Now create the agent 

In [ ]:
from langchain.agents import create_agent

agent = create_agent("openai:gpt-5", tools=tools)

In [ ]:
from langchain_classic.agents import  create_tool_calling_agent

ImportError: cannot import name 'create_tool_calling_agent' from 'langchain.agents' (/home/aljebra/Generative AI tutorial/venv/lib/python3.10/site-packages/langchain/agents/__init__.py)

### To execute the prompt, llm and tools together in the form of chain - use agent

In [ ]:
# Create the agent
agent = create_tool_calling_agent(
    llm=model,         # your Groq Llama3 model
    tools=tools,       # your list of tools
    prompt=prompt      # your prompt template
)

In [ ]:
agent

### Run the Agent

In [ ]:
from langchain.agents import AgentExecutor

In [ ]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
agent_executor.invoke({"input": "what is langsmith"})

In [ ]:
agent_executor.invoke({"input": "what is machine learning"})

In [ ]:
agent_executor.invoke({"input": "what is programming"})

In [ ]:
agent_executor.invoke({"input": "tell me more about programming"})